In [0]:
%sql
-- 1. Creas la tabla de destino vacía (Streaming Table)
CREATE OR REFRESH STREAMING TABLE target_orders;

-- 2. Aplicas la lógica de CDC de forma declarativa
AUTO CDC INTO LIVE.target_orders
FROM STREAM LIVE.orders_raw_cdc
KEYS (order_id)                             -- Llave primaria de negocio para el UPSERT
SEQUENCE BY (timestamp_evento)              -- ¡VITAL! Evalúa matemáticamente qué registro es el más nuevo
IGNORE NULL UPDATES                         -- Mantiene el valor antiguo si el update manda un nulo
STORED AS SCD TYPE 2
APPLY AS DELETE WHEN operacion = 'DELETE';   -- Borra físicamente si se cumple la condición

In [0]:
import dlt

# 1. Declaramos la tabla destino
dlt.create_streaming_table("target_orders")

# 2. Aplicamos la lógica de consolidación
dlt.apply_changes(
    target="target_orders",
    source="orders_raw_cdc",
    keys=["order_id"],
    sequence_by="timestamp_evento",
    ignore_null_updates=True,
    apply_as_deletes="operacion = 'DELETE'",
    stored_as_scd_type=1 # SCD Type 1 sobrescribe; usa 2 para mantener historial (__start_at y __end_at)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# 1. Función que procesará cada micro-lote de forma individual
def upsert_cdc_batch(micro_lote_df, batch_id):
    # Paso A: Eliminar duplicados dentro del propio micro-lote (nos quedamos con el más nuevo)
    window_spec = Window.partitionBy("order_id").orderBy(col("timestamp_evento").desc())
    deduped_batch = micro_lote_df.withColumn("rn", row_number().over(window_spec)).filter("rn = 1").drop("rn")
    
    # Paso B: Registrar temporalmente el lote para poder usar Spark SQL
    deduped_batch.createOrReplaceTempView("current_batch")
    
    # Paso C: Ejecutar el MERGE INTO manual contra la tabla destino Delta
    micro_lote_df.sparkSession.sql("""
        MERGE INTO main.default.target_orders AS target
        USING current_batch AS source
        ON target.order_id = source.order_id
        WHEN MATCHED AND source.operacion = 'DELETE' THEN
          DELETE
        WHEN MATCHED THEN
          UPDATE SET *
        WHEN NOT MATCHED AND source.operacion <> 'DELETE' THEN
          INSERT *
    """)

# 2. Configurar el writeStream utilizando foreachBatch
(spark.readStream
 .format("cloudFiles")
 .option("cloudFiles.format", "json")
 .load("/Volumes/main/default/bronze_cdc")
 .writeStream
 .format("delta")
 .option("checkpointLocation", "/Volumes/main/default/checkpoint_cdc")
 # 💡 AQUÍ SE LE ENCOMIENDA LA LOGICA A TU FUNCIÓN MANUAL
 .foreachBatch(upsert_cdc_batch)
 .trigger(availableNow=True)
 .start()
)